# Delta MERGE Idempotency — Isolated Integration Test

Tests UPDATE, INSERT, unchanged rows, repeatability, and row-level equality using **four synthetic records**. A unique `silver_test` table is created for each run. The notebook never reads or writes `silver.fact_sale`.

**Prerequisites:** A Microsoft Fabric Lakehouse attached as the default Lakehouse with a pre-created `silver_test` schema and an available Spark session. This notebook uses `delta.tables.DeltaTable`.

**Execution:** Run the cells from top to bottom. The last cell drops **only this execution's uniquely named test table** when `CLEANUP_AFTER_TEST=True`. Set to `False` to retain evidence temporarily. If execution stops before cleanup, manually inspect the generated test table. Do not publish Fabric workspace identifiers or credentials.

**Verification status:** Refactored from a user-executed test; **this exact notebook has not yet been executed in Fabric**.

In [ ]:
from decimal import Decimal
from uuid import uuid4
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DecimalType

TEST_SCHEMA = "silver_test"
TEST_PREFIX = "fact_sale_merge_test_"
CLEANUP_AFTER_TEST = True

if not spark.catalog.databaseExists(TEST_SCHEMA):
    raise RuntimeError(f"Missing test schema: {TEST_SCHEMA}. Create it before running.")

RUN_ID = uuid4().hex[:12]
TEST_TABLE = f"{TEST_SCHEMA}.{TEST_PREFIX}{RUN_ID}"
assert TEST_TABLE.startswith("silver_test.fact_sale_merge_test_")
assert not spark.catalog.tableExists(TEST_TABLE)

schema = StructType([
    StructField("SaleKey", LongType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("TotalIncludingTax", DecimalType(18, 2), False),
])

historical_rows = [
    (1, 5, Decimal("100.00")),
    (2, 10, Decimal("200.00")),
    (3, 15, Decimal("300.00")),
]
incoming_rows = [
    (2, 12, Decimal("240.00")),
    (3, 15, Decimal("300.00")),
    (4, 8, Decimal("160.00")),
]

source = spark.createDataFrame(incoming_rows, schema)
initial = spark.createDataFrame(historical_rows, schema)
initial.write.format("delta").mode("errorifexists").saveAsTable(TEST_TABLE)
print("Isolated test table:", TEST_TABLE)
assert spark.table(TEST_TABLE).count() == 3


In [ ]:
CHANGE_CONDITION = (
    "NOT (t.Quantity <=> s.Quantity) "
    "OR NOT (t.TotalIncludingTax <=> s.TotalIncludingTax)"
)
UPDATE_SET = {
    "Quantity": "s.Quantity",
    "TotalIncludingTax": "s.TotalIncludingTax",
}
INSERT_SET = {
    "SaleKey": "s.SaleKey",
    **UPDATE_SET,
}

def run_merge():
    (DeltaTable.forName(spark, TEST_TABLE).alias("t")
        .merge(source.alias("s"), "t.SaleKey = s.SaleKey")
        .whenMatchedUpdate(condition=CHANGE_CONDITION, set=UPDATE_SET)
        .whenNotMatchedInsert(values=INSERT_SET)
        .execute())

def latest_commit():
    return spark.sql(f"DESCRIBE HISTORY {TEST_TABLE}").orderBy("version", ascending=False).first()

run_merge()
first = latest_commit()
assert first["operation"] == "MERGE", first["operation"]
first_metrics = first["operationMetrics"] or {}
assert int(first_metrics.get("numTargetRowsUpdated", -1)) == 1, first_metrics
assert int(first_metrics.get("numTargetRowsInserted", -1)) == 1, first_metrics
assert int(first_metrics.get("numTargetRowsDeleted", -1)) == 0, first_metrics
first_version = first["version"]
print("FIRST MERGE: PASSED; version:", first_version)
print("First MERGE metrics:", first_metrics)


In [ ]:
run_merge()
second = latest_commit()
second_version = second["version"]
assert second_version >= first_version

if second_version > first_version:
    assert second["operation"] == "MERGE"
    second_metrics = second["operationMetrics"] or {}
    assert int(second_metrics.get("numTargetRowsUpdated", -1)) == 0, second_metrics
    assert int(second_metrics.get("numTargetRowsInserted", -1)) == 0, second_metrics
    assert int(second_metrics.get("numTargetRowsDeleted", -1)) == 0, second_metrics
    print("REPEAT MERGE: new no-op transaction; zero row changes")
else:
    print("REPEAT MERGE: no new Delta commit (previous metrics are NOT rerun metrics)")

print("Version before:", first_version, "Version after:", second_version)


In [ ]:
expected = {
    1: (5, Decimal("100.00")),
    2: (12, Decimal("240.00")),
    3: (15, Decimal("300.00")),
    4: (8, Decimal("160.00")),
}
rows = spark.table(TEST_TABLE).select("SaleKey", "Quantity", "TotalIncludingTax").collect()
actual = {r["SaleKey"]: (r["Quantity"], r["TotalIncludingTax"]) for r in rows}
assert len(rows) == len(expected), (len(rows), len(expected))
assert actual == expected, (actual, expected)
print("FINAL ROW-LEVEL STATE: PASSED")
print("Unique test table:", TEST_TABLE)

if CLEANUP_AFTER_TEST:
    if not TEST_TABLE.startswith("silver_test.fact_sale_merge_test_"):
        raise RuntimeError("Unsafe cleanup target")
    spark.sql(f"DROP TABLE {TEST_TABLE}")
    assert not spark.catalog.tableExists(TEST_TABLE)
    print("Isolated test table cleaned up")
else:
    print("Test table retained for inspection:", TEST_TABLE)
